In [6]:
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torch.optim import AdamW
from transformers import (
    RobertaTokenizer, 
    RobertaModel, 
    RobertaConfig,
    get_linear_schedule_with_warmup,

)
from torch.cuda.amp import autocast, GradScaler  # Mixed precision training
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error
import os
import gc
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')


In [7]:

def setup_gpu_optimizations():
    """Configure all GPU optimizations for RTX 2050"""
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    if torch.cuda.is_available():
        print("="*60)
        print("GPU OPTIMIZATION SETUP FOR RTX 2050")
        print("="*60)
        
        gpu_name = torch.cuda.get_device_name(0)
        total_memory = torch.cuda.get_device_properties(0).total_memory / 1024**3
        print(f"GPU: {gpu_name}")
        print(f"Total VRAM: {total_memory:.1f} GB")
        
        # OPTIMIZATION 1: Enable TF32 for Ampere GPIs (2x faster!)
        if torch.cuda.get_device_capability()[0] >= 8:
            torch.backends.cuda.matmul.allow_tf32 = True
            torch.backends.cudnn.allow_tf32 = True
            print("✓ TF32 enabled (faster matrix operations)")
        
        # OPTIMIZATION 2: Enable cudnn benchmark
        torch.backends.cudnn.benchmark = True
        print("✓ cuDNN benchmark enabled")
        
        # OPTIMIZATION 3: Set memory allocation strategy
        os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'max_split_size_mb:128'
        print("✓ Optimized memory allocation strategy")
        
        # OPTIMIZATION 4: Clear cache
        torch.cuda.empty_cache()
        
        # Check initial memory
        print(f"\nInitial GPU memory: {torch.cuda.memory_allocated()/1024**2:.1f} MB")
        
    return device

# Initialize GPU optimizations
device = setup_gpu_optimizations()

GPU OPTIMIZATION SETUP FOR RTX 2050
GPU: NVIDIA GeForce RTX 2050
Total VRAM: 4.0 GB
✓ TF32 enabled (faster matrix operations)
✓ cuDNN benchmark enabled
✓ Optimized memory allocation strategy

Initial GPU memory: 0.0 MB


In [3]:
train_df = pd.read_csv("emotion_intensity_data/train.csv")
dev_df = pd.read_csv("emotion_intensity_data/dev.csv")
test_df = pd.read_csv("emotion_intensity_data/test.csv")

print("\nSample data:\n", train_df.head())


Sample data:
                         id                                               text  \
0  eng_train_track_b_00001                       Colorado, middle of nowhere.   
1  eng_train_track_b_00002  This involved swimming a pretty large lake tha...   
2  eng_train_track_b_00003        It was one of my most shameful experiences.   
3  eng_train_track_b_00004  After all, I had vegetables coming out my ears...   
4  eng_train_track_b_00005                        Then the screaming started.   

   anger  fear  joy  sadness  surprise  
0      0     1    0        0         1  
1      0     2    0        0         0  
2      0     1    0        3         0  
3      0     0    0        0         0  
4      0     3    0        1         2  


In [ ]:
EMOTIONS = ["anger", "fear", "joy", "sadness", "surprise"]
TEXT_COL = "text"